# 13. Pakistan set: overlap checks, scoring and ablation

Runs after `11_final_preregistered_run_osm` and `12_ablation_osm`. In order: the overlap check of the second-round rows against both training corpora; the one-time scoring of the frozen final configuration on the full Pakistan set (S0 to S4, guarded by `SCORED.flag`); the source check of the first-round rows (SC0 to SC3); and the ablation on the primary subset (AX0 to AX3). Every scoring cell first reproduces the saved results before computing anything new.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Colab cell (CPU is enough). Checks the new Pakistan rows against both training corpora before they are used.
# 1. Upload pk_extension_review.csv to Drive geoproject/eval/ first.
# 2. Run. Writes geoproject/eval/pk_extension_overlap.csv and prints the flagged rows.
# A row is flagged when its text, or the source listing it was reworded from, shares >= 50% of its
# content words (token Jaccard, words of 4+ letters) with any training sentence, or when the source
# listing appears verbatim in a training sentence.
import re, collections
import pandas as pd

ROOT = '/content/drive/MyDrive/geoproject'
new = pd.read_csv(f'{ROOT}/eval/pk_extension_review.csv')
tok = lambda s: frozenset(re.findall(r'[a-z]{4,}', str(s).lower()))

def text_col(df):
    for c in ('text', 'sentence', 'description', 'desc', 'content'):
        if c in df.columns: return c
    raise KeyError(f'no text column in {list(df.columns)}')

corpora = {}
osm = pd.read_parquet(f'{ROOT}/osmfinal/osm_extracted.parquet')
corpora['osm_final'] = osm[text_col(osm)].astype(str).tolist()
wv = pd.read_parquet(f'{ROOT}/results_i/weak_labelled.parquet')
corpora['wikivoyage_weak'] = wv[text_col(wv)].astype(str).tolist()
print({k: len(v) for k, v in corpora.items()}, '| text columns:', text_col(osm), text_col(wv))

rows = []
for name, sents in corpora.items():
    T = [tok(s) for s in sents]
    inv = collections.defaultdict(list)
    for i, t in enumerate(T):
        for w in t: inv[w].append(i)
    big = '\n'.join(sents).lower()                                   # one string: substring search runs in C
    for _, r in new.iterrows():
        for field in ('text', 'source_text'):
            q = tok(r[field])
            if not q: continue
            hits = collections.Counter(i for w in q if len(inv[w]) < 5000 for i in inv[w])   # skip very common words
            best = max((len(q & T[i]) / len(q | T[i]), i) for i in hits) if hits else (0.0, -1)
            verbatim = field == 'source_text' and len(str(r[field])) > 40 and str(r[field]).lower()[:80] in big
            if best[0] >= 0.5 or verbatim:
                rows.append({'id': r['id'], 'field': field, 'corpus': name, 'jaccard': round(best[0], 3),
                             'verbatim_source': verbatim, 'train_sentence': sents[best[1]][:200] if best[1] >= 0 else ''})
out = pd.DataFrame(rows)
out.to_csv(f'{ROOT}/eval/pk_extension_overlap.csv', index=False)
print(len(out), 'flags over', out['id'].nunique() if len(out) else 0, 'rows')
print(out.to_string()[:6000] if len(out) else 'no overlap found')


In [ ]:
# ============================================================================
# EXTENDED PAKISTAN SET -- post-hoc evaluation of the FROZEN final OSM pipeline.
# The set was built after the final run, so nothing here is pre-registered and nothing
# is tuned: lambda, tables and vocabulary are the saved osmfinal artefacts.
# Arms: final (Numberbatch + OSM conf), Numberbatch + OSM Bayesian EN, base paper (w2v, Alg. 2).
# The Wikivoyage-trained arms are NOT scored: the overlap check found the source listings of
# 179 kept rows in results_i/weak_labelled.parquet.
# Before running: upload pk_extended_raw.json to Drive geoproject/eval/. Runtime: T4 GPU.
# Writes geoproject/pk_extension/pk_extended_results.json once (SCORED.flag).
# ============================================================================

# ---------------------------------------------------------------- CELL S0 --
# Restore + gates. Helpers copied verbatim from 09_ablation_cells.py (R3 / R4a / R5a / R6 / R7).
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, pickle, re, gc, subprocess
import numpy as np, pandas as pd
from collections import Counter
from scipy import sparse
from scipy.stats import binomtest
ROOT = '/content/drive/MyDrive/geoproject'; OUT = f'{ROOT}/osmfinal'; RES = f'{ROOT}/pk_extension'
os.makedirs(RES, exist_ok=True)
assert not os.path.exists(f'{RES}/SCORED.flag'), 'already scored once; read pk_extended_results.json instead'
sys.path.insert(0, ROOT)
import pipeline_ranker as pr

meta = np.load(f'{OUT}/osm_feat_meta.npz', allow_pickle=True)
PV, CLASSES, NADJ = list(meta['patvocab']), list(meta['classes']), int(meta['n_adj'])
T4 = dict(np.load(f'{OUT}/tables_r4a.npz'))
bz = np.load(f'{OUT}/tables_ben.npz'); T4['bayes_en'] = (bz['mean'] * bz['selected']).astype(np.float32)
ADO, FIN = json.load(open(f'{OUT}/adoption.json')), json.load(open(f'{OUT}/final.json'))
LAM = {'conf': ADO['dev']['numberbatch']['conf']['best_lambda'], 'bayes_en': ADO['dev']['numberbatch']['bayes_en']['best_lambda']}

def norm_rows(M):                                            # R5a, verbatim
    lo, hi = M.min(1, keepdims=True), M.max(1, keepdims=True)
    return np.where(hi > lo, (M - lo) / (hi - lo + 1e-9), M)
def featurize(df, pv, nadj):                                 # R3, verbatim
    pix = {p: j for j, p in enumerate(pv)}; A, P2 = set(pv[:nadj]), set(pv[nadj:])
    ri, ci = [], []
    for r, row in enumerate(df.itertuples(index=False)):
        seen = {pix[a] for a in row.adjs if a in A} | {pix[p] for p in row.pit if p in P2}
        ri += [r] * len(seen); ci += list(seen)
    return sparse.csr_matrix((np.ones(len(ri), np.float32), (ri, ci)), shape=(len(df), len(pv)))
def sim_matrix(R, df):                                       # R5a, abstract nouns dropped
    return np.vstack([R._norm(R.sim([t for t in L if t.lower() not in pr.ABSTRACT])) for L in df.context_nouns]).astype(np.float32)
def table_matrix(R, Xs, M, classes=CLASSES):                 # R5a, verbatim
    agg = Xs @ M
    S = np.zeros((Xs.shape[0], len(R.TAGS)), np.float32)
    for k, c in enumerate(classes):
        if c in R.IX: S[:, R.IX[c]] = agg[:, k]
    has_ = Xs.getnnz(1) > 0
    S[has_] = norm_rows(S[has_]); S[~has_] = 0.0
    return S
def ranks(SIM, TBL, g, lam):                                 # R7, verbatim
    s = SIM if TBL is None or lam == 0 else (1 - lam) * SIM + lam * TBL
    ok = g >= 0; r = np.full(len(g), SIM.shape[1] + 1)
    r[ok] = (s[ok] >= s[np.where(ok)[0], g[ok]][:, None]).sum(1); return r
def mets(r): return {'n': int(len(r)), 'Hits@1': float((r <= 1).mean()), 'Hits@3': float((r <= 3).mean()), 'nDCG@20': float(np.where(r <= 20, 1 / np.log2(r + 1), 0).mean())}
def boot(a, b, n=10000, seed=0):                             # R6, verbatim
    rs = np.random.RandomState(seed); m = len(a); d = np.empty(n); a = a.astype(float); b = b.astype(float)
    for i in range(n):
        ix = rs.randint(0, m, m); d[i] = a[ix].mean() - b[ix].mean()
    return float(a.mean() - b.mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))
def as_df(rows):                                             # R7
    return pd.DataFrame({'context_nouns': [r['context_nouns'] for r in rows], 'tag': [r['gold_tag'] for r in rows],
                         'adjs': [sorted({a.lower() for a in r['adjectives'] if str(a).isalpha()}) for r in rows],
                         'pit': [[it for it in r['transaction'] if it.startswith('amod:')] for r in rows]})

LEX = json.load(open(f'{ROOT}/eval/evalset_106.json'))['lexicon']
WVT = pr.Tables(f'{ROOT}/results_i/i3_elasticnet.npz', f'{ROOT}/results_p/p1_bayesian_en.npz', f'{ROOT}/results_p/p1_feat_meta.npz')
VEC = {'w2v': pickle.load(open(f'{OUT}/w2v_googlenews_osm.pkl', 'rb')), 'numberbatch': pickle.load(open(f'{OUT}/numberbatch_osm.pkl', 'rb'))}
RK = {bb: pr.Ranker(pr.ladder(v), LEX, WVT) for bb, v in VEC.items()}

ARMS = {'final: numberbatch + OSM conf': ('numberbatch', 'conf'), 'numberbatch + OSM Bayesian EN': ('numberbatch', 'bayes_en'),
        'base paper (w2v, Algorithm 2)': ('w2v', None)}
def score(rows):
    df, out = as_df(rows), {}
    for name, (bb, tb) in ARMS.items():
        R = RK[bb]; g = np.array([R.IX.get(t, -1) for t in df.tag])
        TBL = table_matrix(R, featurize(df, PV, NADJ), T4[tb]) if tb else None
        out[name] = ranks(sim_matrix(R, df), TBL, g, LAM[tb] if tb else 0.0)
    return out

X = json.load(open(f'{ROOT}/eval/pk_extended_raw.json'))
evpk = json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json'))
c0 = Counter(r['gold_tag'] for r in evpk); pk3 = [r for r in evpk if c0[r['gold_tag']] >= 3]
orig = {r['id']: r for r in X if r['part'] == 'original'}
assert len(X) == 592 and len(orig) == 199 and len(pk3) == 157
assert all(orig[f"PK{int(r['row']):03d}"]['gold_tag'] == r['gold_tag'] and orig[f"PK{int(r['row']):03d}"]['context_nouns'] == r['context_nouns'] for r in evpk)
def gate(tag):
    S = score(pk3)
    for name in ARMS:
        assert abs(mets(S[name])['Hits@3'] - FIN['results']['pk_n3'][name]['Hits@3']) < 1e-9, (tag, name)
    print(f'gate ({tag}): all 3 arms reproduce final.json on the 157 rows exactly')
gate('saved caches')


In [ ]:
# ---------------------------------------------------------------- CELL S1 --
# Extraction of all 592 English sentences with the same transformer extractor, then the embedding
# caches are extended for the new nouns (R5-cache recipe). Tag vectors are NOT rebuilt: only the
# query-side lookup gets the extra vectors, and the gate is re-run.
if subprocess.run([sys.executable, '-c', 'import en_core_web_trf'], capture_output=True).returncode:
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_trf', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gensim>=4.3.3'], check=True)
import spacy; spacy.prefer_gpu()
import adjective_extractor as ae
PI = ae.PlaceInterpreter(model='en_core_web_trf')
def extract(texts):
    t0 = time.time(); ex = pr.extract_rows(list(texts), PI); print(f'   extracted {len(ex)} texts in {time.time()-t0:.0f}s'); return ex
EX = dict(zip([r['id'] for r in X], extract([r['sentence'] for r in X])))
same = np.mean([EX[i]['adjectives'] == orig[i]['adjectives'] for i in orig])
print(f'   original rows: re-extracted adjectives identical to the saved ones for {same:.1%}')

def parts(s): return [w for w in re.split(r'[\s\-_/]+', s) if w]
need = {p for e in EX.values() for n in e['context_nouns'] for p in [n] + parts(n)}
probe = {bb: {v for s in need for v in pr.variants(s)} - set(VEC[bb]) for bb in VEC}
import gensim.downloader as api
kv = api.load('word2vec-google-news-300'); NEW_W = {p: np.asarray(kv[p], np.float32) for p in probe['w2v'] if p in kv}; del kv; gc.collect()
kv = api.load('conceptnet-numberbatch-17-06-300'); key = lambda p: f'/c/en/{p.lower()}'
NEW_N = {p: np.asarray(kv[key(p)], np.float32) for p in probe['numberbatch'] if key(p) in kv}; del kv; gc.collect()
RK['w2v'].vec = pr.ladder({**VEC['w2v'], **NEW_W}); RK['numberbatch'].vec = pr.ladder({**VEC['numberbatch'], **NEW_N})
print(f'   +{len(NEW_W)} w2v and +{len(NEW_N)} numberbatch vectors for {len(need)} new noun strings')
gate('extended caches')
for bb, R in RK.items():
    cov = np.mean([R.vec(n) is not None for e in EX.values() for n in e['context_nouns']])
    print(f'   {bb:12s} extractor nouns embeddable {cov:.1%}')
miss = sorted({r['gold_tag'] for r in X} - set(RK['numberbatch'].IX) - set(RK['w2v'].IX)); print('   gold tags without a vector:', miss or 'none')


In [ ]:
# ---------------------------------------------------------------- CELL S2 --
# English. A = the pre-registered 157 with supplied context nouns (= final.json);
# everything else uses extractor nouns for every row, since the new rows have no supplied nouns.
auto = lambda r: {**r, **{k: EX[r['id']][k] for k in ('context_nouns', 'adjectives', 'transaction')}}
cnt = Counter(r['gold_tag'] for r in X); n3 = [r for r in X if cnt[r['gold_tag']] >= 3]
pk3_ids = [f"PK{int(r['row']):03d}" for r in pk3]
SETS = {
    'A original 157, supplied nouns (final.json)':       [orig[i] for i in pk3_ids],
    'A2 original 157, extractor nouns':                 [auto(orig[i]) for i in pk3_ids],
    'B extended n>=3, extractor nouns':                 [auto(r) for r in n3],
    'C extended n>=3, new rows only':                   [auto(r) for r in n3 if r['part'] == 'extension'],
    'D extended n>=3, original rows only':              [auto(r) for r in n3 if r['part'] == 'original'],
    'S1 B without rows marked uncertain':               [auto(r) for r in n3 if not r['uncertain']],
    'S2 B without the 4 type-word rows':                [auto(r) for r in n3 if not r.get('type_word_in_text')],
}
RANKS, ROWS = {}, {}
for s, rows in SETS.items():
    RANKS[s] = score(rows); ROWS[s] = [r['id'] for r in rows]
    print(f'{s:46s} n={len(rows):3d} | ' + ' | '.join(f"{n.split(':')[0].split(' (')[0][:22]} H@3 {mets(RANKS[s][n])['Hits@3']:.3f}" for n in ARMS))


In [ ]:
# ---------------------------------------------------------------- CELL S3 --
# Urdu and Roman Urdu through the translation front end (Phase N translators and prompt, verbatim),
# then the same extractor and arms. Each is paired with the English originals of the same rows.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'bitsandbytes>=0.46.1'], check=True)
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, BitsAndBytesConfig
dev = 'cuda'; assert torch.cuda.is_available(), 'switch the runtime to a GPU (T4 is enough)'
nl_tok = AutoTokenizer.from_pretrained('facebook/nllb-200-distilled-600M', src_lang='urd_Arab'); nl = AutoModelForSeq2SeqLM.from_pretrained('facebook/nllb-200-distilled-600M', dtype=torch.float16).to(dev).eval()
def nllb(texts, bs=16):
    out = []
    for i in range(0, len(texts), bs):
        enc = nl_tok(texts[i:i+bs], return_tensors='pt', padding=True, truncation=True, max_length=256).to(dev)
        with torch.no_grad(): gen = nl.generate(**enc, forced_bos_token_id=nl_tok.convert_tokens_to_ids('eng_Latn'), max_new_tokens=160, num_beams=4)
        out += nl_tok.batch_decode(gen, skip_special_tokens=True)
    return out
qw_id = 'Qwen/Qwen2.5-7B-Instruct'
qw_tok = AutoTokenizer.from_pretrained(qw_id); qw = AutoModelForCausalLM.from_pretrained(qw_id, quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4'), device_map='auto').eval()
def llm_translate(texts, lang, bs=8):
    out = []; qw_tok.padding_side = 'left'
    for i in range(0, len(texts), bs):
        msgs = [[{'role': 'system', 'content': 'You are a translator. Translate the user text into natural English. Keep place names as they are. Output only the translation.'}, {'role': 'user', 'content': f'{lang} text: {t}'}] for t in texts[i:i+bs]]
        enc = qw_tok([qw_tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs], return_tensors='pt', padding=True).to(dev)
        with torch.no_grad(): gen = qw.generate(**enc, max_new_tokens=120, do_sample=False)
        out += [s.strip().split('\n')[0] for s in qw_tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)]
    return out

ur = [r for r in X if r['urdu']]; ru = [r for r in X if r['roman_urdu']]
assert len(ur) == 157 and len(ru) == 300
t0 = time.time()
TR = {'urdu_nllb': dict(zip([r['id'] for r in ur], nllb([r['urdu'] for r in ur]))),
      'roman_llm': dict(zip([r['id'] for r in ru], llm_translate([r['roman_urdu'] for r in ru], 'Roman Urdu (Urdu written in Latin letters, possibly mixed with English)')))}
print(f'translated {len(ur)} Urdu + {len(ru)} Roman Urdu in {time.time()-t0:.0f}s')
EXT = {k: dict(zip(v, extract(list(v.values())))) for k, v in TR.items()}
via = lambda k: (lambda r: {**r, **{f: EXT[k][r['id']][f] for f in ('context_nouns', 'adjectives', 'transaction')}})
ORIGINS = {'author': 'written by author', 'drafted': 'drafted, '}
LANG = {
    'U urdu 157 -> NLLB':                 [via('urdu_nllb')(r) for r in ur],
    'U-en same 157 rows, English':        [auto(r) for r in ur],
    'R roman 300 -> LLM':                 [via('roman_llm')(r) for r in ru],
    'R-en same 300 rows, English':        [auto(r) for r in ru],
    'R1 roman, author-written 60 -> LLM': [via('roman_llm')(r) for r in ru if r['roman_origin'] == ORIGINS['author']],
    'R1-en same 60 rows, English':        [auto(r) for r in ru if r['roman_origin'] == ORIGINS['author']],
    'R2 roman, drafted 240 -> LLM':       [via('roman_llm')(r) for r in ru if r['roman_origin'].startswith(ORIGINS['drafted'])],
    'R2-en same 240 rows, English':       [auto(r) for r in ru if r['roman_origin'].startswith(ORIGINS['drafted'])],
}
assert len(LANG['R1 roman, author-written 60 -> LLM']) == 60 and len(LANG['R2 roman, drafted 240 -> LLM']) == 240
for s, rows in LANG.items():
    RANKS[s] = score(rows); ROWS[s] = [r['id'] for r in rows]
    print(f'{s:46s} n={len(rows):3d} | ' + ' | '.join(f"{n.split(':')[0].split(' (')[0][:22]} H@3 {mets(RANKS[s][n])['Hits@3']:.3f}" for n in ARMS))


In [ ]:
# ---------------------------------------------------------------- CELL S4 --
# Comparisons (paired bootstrap 95% CI on the Hits@3 difference, exact McNemar, Holm over this file)
# and the one write.
def holm(ps):
    o = np.argsort(ps); adj = np.empty(len(ps)); run = 0.0
    for k, i in enumerate(o): run = max(run, (len(ps) - k) * ps[i]); adj[i] = min(1.0, run)
    return adj.tolist()
F, BEN, B0 = list(ARMS)
CMP = []
for s in RANKS:
    for a, b in ((F, B0), (BEN, F)):
        ha, hb = RANKS[s][a] <= 3, RANKS[s][b] <= 3
        d, lo, hi = boot(ha, hb); ao, bo = int((ha & ~hb).sum()), int((hb & ~ha).sum())
        CMP.append({'set': s, 'a': a, 'b': b, 'delta_hits3': d, 'ci95': [lo, hi], 'a_only': ao, 'b_only': bo,
                    'p': float(binomtest(min(ao, bo), ao + bo, 0.5).pvalue) if ao + bo else 1.0})
for c, p in zip(CMP, holm([c['p'] for c in CMP])): c['p_holm'] = p
for s, (en, tr) in {'urdu': ('U-en same 157 rows, English', 'U urdu 157 -> NLLB'), 'roman': ('R-en same 300 rows, English', 'R roman 300 -> LLM'),
                    'roman_author': ('R1-en same 60 rows, English', 'R1 roman, author-written 60 -> LLM'),
                    'roman_drafted': ('R2-en same 240 rows, English', 'R2 roman, drafted 240 -> LLM')}.items():
    d, lo, hi = boot(RANKS[tr][F] <= 3, RANKS[en][F] <= 3)
    CMP.append({'set': f'translation loss, {s}', 'a': tr, 'b': en, 'arm': F, 'delta_hits3': d, 'ci95': [lo, hi]})
for c in CMP:
    print(f"{c['set'][:40]:40s} {c['a'][:28]:28s} vs {c['b'][:28]:28s} d {c['delta_hits3']:+.3f} [{c['ci95'][0]:+.3f}, {c['ci95'][1]:+.3f}]" + (f"  p {c['p']:.4f} holm {c['p_holm']:.4f}" if 'p' in c else ''))

OUTJ = {'note': ('Post-hoc evaluation of the frozen final OSM pipeline on the extended Pakistan set (built after the final run). '
                 'Set A reproduces final.json; all other sets use extractor nouns. Wikivoyage-trained arms are not scored (source overlap).'),
        'lambda': LAM, 'sets': {s: {'n': len(ROWS[s]), 'ids': ROWS[s], **{n: mets(RANKS[s][n]) for n in ARMS}} for s in RANKS},
        'ranks': {s: {n: RANKS[s][n].tolist() for n in ARMS} for s in RANKS}, 'comparisons': CMP,
        'translations': TR, 'models': {'nllb': 'facebook/nllb-200-distilled-600M (urd_Arab->eng_Latn, 4 beams)', 'llm': qw_id + ' (4-bit, greedy)'},
        'extractor': 'en_core_web_trf via adjective_extractor.PlaceInterpreter', 'extra_vectors': {'w2v': len(NEW_W), 'numberbatch': len(NEW_N)},
        'input': 'eval/pk_extended_raw.json', 'n3_tags': sorted(t for t, c in cnt.items() if c >= 3)}
json.dump(OUTJ, open(f'{RES}/pk_extended_results.json', 'w'), ensure_ascii=False, indent=1)
open(f'{RES}/SCORED.flag', 'w').write(time.strftime('%Y-%m-%d %H:%M:%S'))
print('wrote', f'{RES}/pk_extended_results.json', '+ SCORED.flag')


In [ ]:
# ============================================================================
# SOURCE CHECK of the first-round (original 199) Pakistan rows, and EXTRACTION STATISTICS of the full set.
# Post hoc; nothing is refitted. Writes pk_extension/original_source_check.{csv,json} and
# pk_extension/extended_extraction_stats.json.
# ============================================================================

# ---------------------------------------------------------------- CELL SC0 --
# the two training corpora and the first-round rows
import json, sys, subprocess, pandas as pd
from collections import Counter
ROOT = '/content/drive/MyDrive/geoproject'
wv = pd.read_parquet(f'{ROOT}/results_i/weak_labelled.parquet')
osm = pd.read_parquet(f'{ROOT}/osmfinal/osm_extracted.parquet')
print('weak:', wv.shape, list(wv.columns)); print(wv[['source', 'place_name', 'split']].head(3).to_string())
print('osm:', osm.shape, list(osm.columns))
evpk = json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json'))
print(Counter(r['source'] for r in evpk))


In [ ]:
# ---------------------------------------------------------------- CELL SC1 --
# OSM rows: is the element itself in the final OSM corpus?  Wikivoyage rows: is the venue's own listing
# in the weakly labelled Wikivoyage corpus (Pakistani pages only, matched on the venue name)?
import re, numpy as np
sys.path.insert(0, ROOT)
pk_rows = pd.read_parquet(f'{ROOT}/results_m/pk_from_corpus.parquet'); print('pk_from_corpus:', pk_rows.shape, list(pk_rows.columns))
norm = lambda s: re.sub(r'[^a-z0-9]+', ' ', str(s).lower()).strip()
tok = lambda s: set(re.findall(r'[a-z]{4,}', str(s).lower()))
key = 'sent_id' if 'sent_id' in pk_rows.columns else None
wpk = wv[wv.sent_id.isin(pk_rows.sent_id)] if key else wv[wv.place_name.map(norm).isin({norm(r['venue']) for r in evpk})]
print('weak-labelled rows on Pakistani pages:', len(wpk), Counter(wpk.split))
by_name = {n: g for n, g in wpk.groupby(wpk.place_name.map(norm))}
osm_ids = dict(zip(osm.osm_id, osm.split))
c3 = Counter(r['gold_tag'] for r in evpk)
REC = []
for r in evpk:
    rec = {'row': r['row'], 'source': r['source'], 'gold': r['gold_tag'], 'n3': c3[r['gold_tag']] >= 3}
    if r['source'] == 'osm':
        rec['in_corpus'] = osm_ids.get(r['source_id'].replace('osm:', ''), '')
    elif r['source'] == 'wikivoyage':
        g = by_name.get(norm(r['venue']))
        if g is not None:
            rec['in_corpus'] = ','.join(sorted(set(g.split)))
            rec['same_weak_tag'] = bool((g.weak_tag == r['gold_tag']).any())
            rec['max_jaccard'] = round(max(len(tok(r['sentence']) & tok(t)) / max(1, len(tok(r['sentence']) | tok(t))) for t in g.text), 3)
            rec['n_sent'] = len(g)
    REC.append(rec)
S = pd.DataFrame(REC).fillna({'in_corpus': ''})
for src in ('wikivoyage', 'osm'):
    s = S[S.source == src]
    print(f"{src}: {len(s)} rows | source in corpus: {int((s.in_corpus != '').sum())} (train: {int(s.in_corpus.str.contains('train').sum())}) | "
          f"n>=3 rows with source in train: {int((s.n3 & s.in_corpus.str.contains('train')).sum())}"
          + (f" | weak tag = gold for {int(s.same_weak_tag.fillna(False).sum())}" if src == 'wikivoyage' else ''))
S.to_csv(f'{ROOT}/pk_extension/original_source_check.csv', index=False)
print(S[S.in_corpus != ''].head(12).to_string())


In [ ]:
# ---------------------------------------------------------------- CELL SC2 --
# Effect on the PRELIMINARY (Wikivoyage-trained) Pakistan results of Section 5.4.9: rescore the 157 rows
# without the rows whose source is in the weakly labelled TRAIN split. Gate first: full 157 must give
# the published 0.618 (W2V + table) and 0.510 (W2V nouns only).
import pipeline_ranker as pr
from scipy.stats import binomtest
print('weak corpus sources:', Counter(wv.source))
for r in evpk:                                                     # OSM eval rows: also look for their name on Pakistani pages
    if r['source'] == 'osm':
        g = by_name.get(norm(r['venue']))
        if g is not None: S.loc[S.row == r['row'], 'weak_split'] = ','.join(sorted(set(g.split)))
S['weak_train'] = (S.source == 'wikivoyage') & S.in_corpus.str.contains('train') | S.get('weak_split', pd.Series('', index=S.index)).fillna('').str.contains('train')
pk3 = [r for r in evpk if c3[r['gold_tag']] >= 3]
drop = set(S[S.weak_train & S.n3].row)
RKP = {'W2V': pr.load_default(ROOT, 'w2v'), 'Numberbatch': pr.load_default(ROOT, 'numberbatch')}
def h3(rk, rows, lam): return float((rk.ranks(rows, lam=lam) <= 3).mean())
full = {'W2V + table': h3(RKP['W2V'], pk3, 0.35), 'W2V nouns only': h3(RKP['W2V'], pk3, 0), 'Numberbatch + table': h3(RKP['Numberbatch'], pk3, 0.35)}
print('full 157:', {k: round(v, 3) for k, v in full.items()})
assert abs(full['W2V + table'] - 0.6178343949044586) < 1e-9 and abs(full['W2V nouns only'] - 0.5095541401273885) < 1e-9, 'gate failed'
rest = [r for r in pk3 if r['row'] not in drop]
cut = {'W2V + table': h3(RKP['W2V'], rest, 0.35), 'W2V nouns only': h3(RKP['W2V'], rest, 0), 'Numberbatch + table': h3(RKP['Numberbatch'], rest, 0.35)}
print(f'without the {len(drop)} rows whose source is in weak train ({len(rest)} left):', {k: round(v, 3) for k, v in cut.items()})
a, b = RKP['W2V'].ranks(rest, lam=0.35) <= 3, RKP['W2V'].ranks(rest, lam=0) <= 3
ao, bo = int((a & ~b).sum()), int((b & ~a).sum())
print('transfer vs base on the rest: d', round(a.mean() - b.mean(), 3), '| improved/worsened', ao, bo, '| McNemar p', round(binomtest(min(ao, bo), ao + bo, .5).pvalue, 4))
osm_in = S[(S.source == 'osm') & (S.in_corpus != '')]; print('OSM eval rows in the final OSM corpus:', osm_in[['row', 'gold', 'n3', 'in_corpus']].to_dict('records'))
json.dump({'wikivoyage_rows': 164, 'wikivoyage_source_in_weak': int((S[S.source == 'wikivoyage'].in_corpus != '').sum()),
           'wikivoyage_source_in_weak_train': int(((S.source == 'wikivoyage') & S.in_corpus.str.contains('train')).sum()),
           'n3_rows_dropped': len(drop), 'dropped_rows': sorted(int(x) for x in drop), 'full_157': full, 'without_dropped': cut,
           'transfer_vs_base_rest': {'delta': float(a.mean() - b.mean()), 'improved': ao, 'worsened': bo, 'p': float(binomtest(min(ao, bo), ao + bo, .5).pvalue)},
           'osm_rows_in_final_corpus': osm_in[['row', 'gold', 'n3', 'in_corpus']].to_dict('records'),
           'method': 'venue name matched (normalised) against place_name of weak-labelled rows on Pakistani pages (results_m/pk_from_corpus); OSM by element id. Name matching is a lower bound.'},
          open(f'{ROOT}/pk_extension/original_source_check.json', 'w'), indent=1, default=str)
print('wrote pk_extension/original_source_check.json')


In [ ]:
# ---------------------------------------------------------------- CELL SC3 --
# Tables 4.2 / 4.4 / 4.8: extraction statistics for the full set (extractor only, nothing is scored).
# Runs in a fresh Python process (a kernel whose spaCy was imported before the trf install cannot load it).
# Gate: the same statistics on the SAVED first-round extraction must reproduce Table 4.8's first-round column.
SRC = r'''
import json, sys, numpy as np, spacy
ROOT = '/content/drive/MyDrive/geoproject'; sys.path.insert(0, ROOT)
import pipeline_ranker as pr, adjective_extractor as ae
spacy.prefer_gpu(); PI = ae.PlaceInterpreter(model='en_core_web_trf')
X = json.load(open(f'{ROOT}/eval/pk_extended_raw.json')); evpk = json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json'))
EX = dict(zip([r['id'] for r in X], pr.extract_rows([r['sentence'] for r in X], PI)))
def stats(rows):
    pit = [[t for t in r['transaction'] if t.startswith('amod:')] for r in rows]
    return {'Sentences': len(rows), 'Nouns per sentence (this work)': round(float(np.mean([len(r['ours_nouns']) for r in rows])), 2),
            'Adjectives found (total)': int(sum(len(r['adjectives']) for r in rows)),
            'Adjectives per sentence': round(float(np.mean([len(r['adjectives']) for r in rows])), 2),
            'Sentences with ≥1 adjective': f"{100 * np.mean([len(r['adjectives']) > 0 for r in rows]):.1f}%",
            'Adjective–noun pairs per sentence': round(float(np.mean([len(p) for p in pit])), 2),
            'Sentences where a direct tag hint fires': f"{100 * np.mean([len(r['tag_hints']) > 0 for r in rows]):.1f}%"}
out = {'saved_original_199': stats(evpk), 'fresh_original_199': stats([EX[f"PK{int(r['row']):03d}"] for r in evpk]),
       'fresh_new_393': stats([EX[r['id']] for r in X if r['part'] == 'extension']), 'fresh_extended_592': stats(list(EX.values())),
       'samples': {i: {'gold': next(x for x in X if x['id'] == i)['gold_tag'], 'sentence': next(x for x in X if x['id'] == i)['sentence'],
                       'nouns': EX[i]['ours_nouns'], 'adjectives': EX[i]['adjectives'],
                       'pairs': [t[5:] for t in EX[i]['transaction'] if t.startswith('amod:')]} for i in ('PK249', 'PK262', 'PK300', 'PK430')}}
json.dump(out, open(f'{ROOT}/pk_extension/extended_extraction_stats.json', 'w'), ensure_ascii=False, indent=1)
print(json.dumps(out, ensure_ascii=False, indent=1))
'''
open('/content/ext_stats.py', 'w').write(SRC)
r = subprocess.run([sys.executable, '/content/ext_stats.py'], capture_output=True, text=True)
print(r.stdout[-6000:]); print(r.stderr[-1500:] if r.returncode else '')


In [ ]:
# ============================================================================
# ABLATION on the EXTENDED Pakistan set (528 rows, extractor nouns) -- post-hoc, like the ablation itself.
# Same arms, helpers and lambda rule as 09_ablation_cells.py; every arm must reproduce ablation.json on
# OSM dev and the 106 set before its Pakistan figure is used. The OSM test split is never read.
# F arms (Wikivoyage tables) are not scored: the new rows' sources are in that training text.
# Writes osmfinal/ablation_pk_extended.json only (ablation.json is untouched).
# ============================================================================

# ---------------------------------------------------------------- CELL AX0 --
# 1. extractor output for the 592 rows, in a fresh process (spaCy trf registry), saved next to the set
import os, sys, json, time, pickle, re, gc, subprocess
import numpy as np, pandas as pd
from collections import Counter
from scipy import sparse
from scipy.stats import binomtest
ROOT = '/content/drive/MyDrive/geoproject'; OUT = f'{ROOT}/osmfinal'; RES = f'{ROOT}/pk_extension'
EXT = f'{RES}/pk_extended_extracted.json'
if not os.path.exists(EXT):
    open('/content/ext_dump.py', 'w').write(r'''
import json, sys, spacy
ROOT = '/content/drive/MyDrive/geoproject'; sys.path.insert(0, ROOT)
import pipeline_ranker as pr, adjective_extractor as ae
spacy.prefer_gpu(); PI = ae.PlaceInterpreter(model='en_core_web_trf')
X = json.load(open(f'{ROOT}/eval/pk_extended_raw.json'))
ex = pr.extract_rows([r['sentence'] for r in X], PI)
json.dump({r['id']: {k: e[k] for k in ('context_nouns', 'adjectives', 'transaction', 'tag_hints')} for r, e in zip(X, ex)},
          open(f'{ROOT}/pk_extension/pk_extended_extracted.json', 'w'), ensure_ascii=False)
''')
    r = subprocess.run([sys.executable, '/content/ext_dump.py'], capture_output=True, text=True); assert r.returncode == 0, r.stderr[-2000:]
EX = json.load(open(EXT)); print('extractor rows:', len(EX))
sys.path.insert(0, ROOT)
import pipeline_ranker as pr

# 2. restore + gates 1-3 (verbatim from AB0)
D = pd.read_parquet(f'{OUT}/osm_extracted.parquet')
for c in ('adjectives', 'transaction', 'context_nouns'): D[c] = D[c].map(json.loads)
D['adjs'] = D['adjectives'].map(lambda L: sorted({a.lower() for a in L if str(a).isalpha()}))
D['pit']  = D['transaction'].map(lambda L: [it for it in L if it.startswith('amod:')])
tr, dv = D[D.split == 'train'], D[D.split == 'dev']
meta = np.load(f'{OUT}/osm_feat_meta.npz', allow_pickle=True)
PV, CLASSES, NADJ, MIN_SUP = list(meta['patvocab']), list(meta['classes']), int(meta['n_adj']), float(meta['min_support'])
CIX = {c: k for k, c in enumerate(CLASSES)}; K = len(CLASSES)
ytr, ydv = tr.tag.map(CIX).values, dv.tag.map(CIX).values
assert (ytr == meta['ytr']).all() and (ydv == meta['ydv']).all(), 'labels differ from R3'
def _ok(p):
    body = p.split(':', 1)[1] if ':' in p else p
    return all(len(t) > 1 and t.isalnum() for t in body.split('_'))
def mine(train, min_sup):
    n_min = min_sup * len(train)
    c1 = Counter(a for L in train.adjs for a in L); c2 = Counter(p for L in train.pit for p in L)
    fa = sorted(a for a, c in c1.items() if c >= n_min and _ok(a))
    fp = sorted(p for p, c in c2.items() if c >= n_min and _ok(p))
    return fa + fp, len(fa)
def featurize(df, pv, nadj):
    pix = {p: j for j, p in enumerate(pv)}; A, P2 = set(pv[:nadj]), set(pv[nadj:])
    ri, ci = [], []
    for r, row in enumerate(df.itertuples(index=False)):
        seen = {pix[a] for a in row.adjs if a in A} | {pix[p] for p in row.pit if p in P2}
        ri += [r] * len(seen); ci += list(seen)
    return sparse.csr_matrix((np.ones(len(ri), np.float32), (ri, ci)), shape=(len(df), len(pv)))
def assoc(Xt, yt, k):
    Cnt = np.zeros((Xt.shape[1], k)); r_, c_ = Xt.nonzero(); np.add.at(Cnt, (c_, yt[r_]), 1)
    Ntot = Cnt.sum(); rt = Cnt.sum(1, keepdims=True); ct = Cnt.sum(0, keepdims=True)
    conf = np.divide(Cnt, rt, out=np.zeros_like(Cnt), where=rt > 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        lift = (Cnt / Ntot) / ((rt / Ntot) * (ct / Ntot))
    return conf.astype(np.float32), np.where(np.isfinite(lift) & (lift > 1), np.log(lift), 0.0).astype(np.float32)
pv_, nadj_ = mine(tr, MIN_SUP); assert pv_ == PV and nadj_ == NADJ
Xtr, Xdv = featurize(tr, PV, NADJ), featurize(dv, PV, NADJ)
for s, Xs in (('train', Xtr), ('dev', Xdv)): assert (Xs != sparse.load_npz(f'{OUT}/X_{s}.npz')).nnz == 0
T4 = dict(np.load(f'{OUT}/tables_r4a.npz'))
bz = np.load(f'{OUT}/tables_ben.npz'); T4['bayes_en'] = (bz['mean'] * bz['selected']).astype(np.float32)
CONF, LOGLIFT = assoc(Xtr, ytr, K); assert np.allclose(CONF, T4['conf']) and np.allclose(LOGLIFT, T4['loglift'])
print('gates 1-3 passed')

LEX = json.load(open(f'{ROOT}/eval/evalset_106.json'))['lexicon']
WVT = pr.Tables(f'{ROOT}/results_i/i3_elasticnet.npz', f'{ROOT}/results_p/p1_bayesian_en.npz', f'{ROOT}/results_p/p1_feat_meta.npz')
VEC = {'w2v': pickle.load(open(f'{OUT}/w2v_googlenews_osm.pkl', 'rb')), 'numberbatch': pickle.load(open(f'{OUT}/numberbatch_osm.pkl', 'rb'))}
RK = {bb: pr.Ranker(pr.ladder(v), LEX, WVT) for bb, v in VEC.items()}

# 3. query-side vectors for the new nouns (R5-cache recipe, as in colab_score_extended.py S1); tag vectors unchanged
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gensim>=4.3.3'], check=True)
def parts(s): return [w for w in re.split(r'[\s\-_/]+', s) if w]
need = {p for e in EX.values() for n in e['context_nouns'] for p in [n] + parts(n)}
probe = {bb: {v for s in need for v in pr.variants(s)} - set(VEC[bb]) for bb in VEC}
import gensim.downloader as api
kv = api.load('word2vec-google-news-300'); NEW_W = {p: np.asarray(kv[p], np.float32) for p in probe['w2v'] if p in kv}; del kv; gc.collect()
kv = api.load('conceptnet-numberbatch-17-06-300'); key = lambda p: f'/c/en/{p.lower()}'
NEW_N = {p: np.asarray(kv[key(p)], np.float32) for p in probe['numberbatch'] if key(p) in kv}; del kv; gc.collect()
RK['w2v'].vec = pr.ladder({**VEC['w2v'], **NEW_W}); RK['numberbatch'].vec = pr.ladder({**VEC['numberbatch'], **NEW_N})
print(f'+{len(NEW_W)} w2v / +{len(NEW_N)} numberbatch query vectors')

# 4. helpers (verbatim) and the three sets
def norm_rows(M):
    lo, hi = M.min(1, keepdims=True), M.max(1, keepdims=True)
    return np.where(hi > lo, (M - lo) / (hi - lo + 1e-9), M)
def sim_matrix(R, df, drop_abstract=True, extra=None):
    out = []
    for i, L in enumerate(df.context_nouns):
        terms = [t for t in L if not drop_abstract or t.lower() not in pr.ABSTRACT]
        if extra is not None: terms = terms + list(extra[i])
        out.append(R._norm(R.sim(terms)))
    return np.vstack(out).astype(np.float32)
def table_matrix(R, Xs, M, classes=CLASSES):
    agg = Xs @ M
    S = np.zeros((Xs.shape[0], len(R.TAGS)), np.float32)
    for k, c in enumerate(classes):
        if c in R.IX: S[:, R.IX[c]] = agg[:, k]
    has_ = Xs.getnnz(1) > 0
    S[has_] = norm_rows(S[has_]); S[~has_] = 0.0
    return S
def gold_idx(R, df): return np.array([R.IX.get(t, -1) for t in df.tag])
def ranks(SIM, TBL, g, lam):
    s = SIM if TBL is None or lam == 0 else (1 - lam) * SIM + lam * TBL
    ok = g >= 0; r = np.full(len(g), SIM.shape[1] + 1)
    r[ok] = (s[ok] >= s[np.where(ok)[0], g[ok]][:, None]).sum(1); return r
def mets(r): return {'Hits@1': float((r <= 1).mean()), 'Hits@3': float((r <= 3).mean()), 'nDCG@20': float(np.where(r <= 20, 1 / np.log2(r + 1), 0).mean())}
LAMS = [round(x, 2) for x in np.arange(0.0, 0.85, 0.05)]
def pick(SIM, TBL, g):
    curve = {l: mets(ranks(SIM, TBL, g, l)) for l in LAMS}
    return max(LAMS, key=lambda l: (curve[l]['Hits@3'], curve[l]['nDCG@20']))
def boot(a, b, n=10000, seed=0):
    rs = np.random.RandomState(seed); m = len(a); d = np.empty(n); a = a.astype(float); b = b.astype(float)
    for i in range(n):
        ix = rs.randint(0, m, m); d[i] = a[ix].mean() - b[ix].mean()
    return float(a.mean() - b.mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))
def as_df(rows):
    return pd.DataFrame({'context_nouns': [r['context_nouns'] for r in rows], 'tag': [r['gold_tag'] for r in rows],
                         'adjs': [sorted({a.lower() for a in r.get('adjectives', []) if str(a).isalpha()}) for r in rows],
                         'pit': [[it for it in r.get('transaction', []) if it.startswith('amod:')] for r in rows],
                         'adjectives': [r.get('adjectives', []) for r in rows], 'transaction': [r.get('transaction', []) for r in rows]})
ev106 = json.load(open(f'{ROOT}/eval/evalset_106_extracted.json')); ev106 = ev106['rows'] if isinstance(ev106, dict) else ev106
X = json.load(open(f'{ROOT}/eval/pk_extended_raw.json')); cnt = Counter(r['gold_tag'] for r in X)
pkx = [{**r, **{k: EX[r['id']][k] for k in ('context_nouns', 'adjectives', 'transaction')}} for r in X if cnt[r['gold_tag']] >= 3]
assert len(pkx) == 528
SETS = {'osm_dev': dv, 'eval_106': as_df(ev106), 'pk_ext': as_df(pkx)}

ARMS, RANKS, SIMC = {}, {}, {}
def sims(bb, s, mode):
    k = (bb, s, mode)
    if k not in SIMC:
        R, df = RK[bb], SETS[s]
        SIMC[k] = (sim_matrix(R, df, drop_abstract=False) if mode == 'keep_abstract' else
                   sim_matrix(R, df, extra=list(df.adjectives)) if mode == 'naive' else sim_matrix(R, df))
    return SIMC[k]
def osm_table(pv, nadj, M, cols=None):
    def f(R, df):
        Xs = featurize(df, pv, nadj)
        return table_matrix(R, Xs if cols is None else Xs[:, cols], M if cols is None else M[cols])
    return f
def arm(name, bb, mode='nouns', tbl=None, sets=None):
    sets = sets or SETS; R = RK[bb]
    g = {s: gold_idx(R, sets[s]) for s in sets}
    TB = {s: (tbl(R, sets[s]) if tbl else None) for s in sets}
    S = {s: (sims(bb, s, mode) if sets is SETS else sim_matrix(R, sets[s], drop_abstract=(mode != 'keep_abstract'))) for s in sets}
    lam = pick(S['osm_dev'], TB['osm_dev'], g['osm_dev']) if tbl else 0.0
    RANKS[name] = {s: ranks(S[s], TB[s], g[s], lam) for s in sets}
    ARMS[name] = {'backbone': bb, 'similarity': mode, 'lambda': lam, **{s: mets(RANKS[name][s]) for s in sets}}
    print(f'{name:44s} lam {lam:.2f} | ' + ' | '.join(f"{s} H@3 {ARMS[name][s]['Hits@3']:.3f}" for s in sets))
AB = json.load(open(f'{OUT}/ablation.json'))
def gate_arm(name):
    a, b = ARMS[name], AB['arms'][name]
    assert a['lambda'] == b['lambda'], (name, 'lambda')
    for s in ('osm_dev', 'eval_106'): assert abs(a[s]['Hits@3'] - b[s]['Hits@3']) < 1e-9, (name, s)


In [ ]:
# ---------------------------------------------------------------- CELL AX1 --
# A, B and C arms, each gated against ablation.json on dev and the 106 set
arm('final: numberbatch + conf (657)', 'numberbatch', tbl=osm_table(PV, NADJ, T4['conf']))
arm('base Alg.2 + abstract removal (w2v)', 'w2v')
cols_adj, cols_pair = np.arange(NADJ), np.arange(NADJ, len(PV))
arm('A0 base Alg.2, abstract nouns kept (w2v)',        'w2v', mode='keep_abstract')
arm('A2 + adjectives appended to the query (w2v)',     'w2v', mode='naive')
arm('A3 + adjective patterns only, conf (w2v)',        'w2v', tbl=osm_table(PV, NADJ, T4['conf'], cols_adj))
arm('A4 + adjective-noun pairs = 657 patterns (w2v)',  'w2v', tbl=osm_table(PV, NADJ, T4['conf']))
arm('B1 final, adjective patterns only',               'numberbatch', tbl=osm_table(PV, NADJ, T4['conf'], cols_adj))
arm('B2 final, pair patterns only',                    'numberbatch', tbl=osm_table(PV, NADJ, T4['conf'], cols_pair))
arm('B3 final, log-lift instead of confidence',        'numberbatch', tbl=osm_table(PV, NADJ, T4['loglift']))
arm('B4 final, frequentist EN (C=0.5)',                'numberbatch', tbl=osm_table(PV, NADJ, T4['fen_C0.5']))
arm('B5 final, Bayesian EN',                           'numberbatch', tbl=osm_table(PV, NADJ, T4['bayes_en']))
arm('B6 final, nouns only (no table)',                 'numberbatch')
arm('B7 final, adjectives appended instead of table',  'numberbatch', mode='naive')
SUPPORT = {}
for ms in (0.002, 0.001, 0.0005, 0.0002, 0.0001):
    pv_s, nadj_s = mine(tr, ms); conf_s, _ = assoc(featurize(tr, pv_s, nadj_s), ytr, K)
    name = f'C support {ms:g}'; arm(name, 'numberbatch', tbl=osm_table(pv_s, nadj_s, conf_s))
    SUPPORT[name] = {'min_support': ms, 'patterns': len(pv_s), 'adjectives': nadj_s, 'pairs': len(pv_s) - nadj_s}
for name in ARMS: gate_arm(name)
print(f'gate: all {len(ARMS)} arms reproduce ablation.json on OSM dev and the 106 set exactly')


In [ ]:
# ---------------------------------------------------------------- CELL AX2 --
# D small parser: OSM rows and the extended rows get nouns, adjectives and pairs from en_core_web_sm
# (the extended rows have no supplied nouns, so all three come from the parser); 106 rows as in AB3.
if subprocess.run([sys.executable, '-c', 'import en_core_web_sm'], capture_output=True).returncode:
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'], check=True)
import spacy, adjective_extractor as ae
I_sm = ae.PlaceInterpreter(model='en_core_web_sm')
sub = D[D.split.isin(['train', 'dev'])].reset_index(drop=True)
t0 = time.time(); rows_sm = pr.extract_rows(list(sub.text), I_sm, batch_size=256); print(f'small parser: {len(rows_sm):,} OSM texts in {time.time()-t0:.0f}s')
SM = pd.DataFrame({'split': sub.split, 'tag': sub.tag, 'context_nouns': [r['context_nouns'] for r in rows_sm],
                   'adjectives': [r['adjectives'] for r in rows_sm], 'transaction': [r['transaction'] for r in rows_sm]})
SM['adjs'] = SM['adjectives'].map(lambda L: sorted({a.lower() for a in L if str(a).isalpha()}))
SM['pit']  = SM['transaction'].map(lambda L: [it for it in L if it.startswith('amod:')])
sm_tr, sm_dv = SM[SM.split == 'train'], SM[SM.split == 'dev']
pv_sm, nadj_sm = mine(sm_tr, MIN_SUP); conf_sm, _ = assoc(featurize(sm_tr, pv_sm, nadj_sm), sm_tr.tag.map(CIX).values, K)
ex106 = pr.extract_rows([r.get('sentence') or r.get('text') for r in ev106], I_sm)
expk = pr.extract_rows([r['sentence'] for r in pkx], I_sm)
SETS_SM = {'osm_dev': sm_dv,
           'eval_106': as_df([{**r, 'adjectives': e['adjectives'], 'transaction': e['transaction']} for r, e in zip(ev106, ex106)]),
           'pk_ext': as_df([{**r, 'context_nouns': e['context_nouns'], 'adjectives': e['adjectives'], 'transaction': e['transaction']} for r, e in zip(pkx, expk)])}
arm('D final, small parser (en_core_web_sm)', 'numberbatch', tbl=osm_table(pv_sm, nadj_sm, conf_sm), sets=SETS_SM)
gate_arm('D final, small parser (en_core_web_sm)')
print('gate: D reproduces ablation.json on dev and the 106 set')


In [ ]:
# ---------------------------------------------------------------- CELL AX3 --
# comparisons on pk_ext (same pairs as AB6 minus the F corpus pairs), Holm within this set's family
F_, A1 = 'final: numberbatch + conf (657)', 'base Alg.2 + abstract removal (w2v)'
PAIRS = [('A1 abstract-noun removal',         A1, 'A0 base Alg.2, abstract nouns kept (w2v)'),
         ('A2 adjectives appended to query',  'A2 + adjectives appended to the query (w2v)', A1),
         ('A3 adjective patterns (table)',    'A3 + adjective patterns only, conf (w2v)', A1),
         ('A4 + pair patterns',               'A4 + adjective-noun pairs = 657 patterns (w2v)', 'A3 + adjective patterns only, conf (w2v)'),
         ('A5 Numberbatch backbone',          F_, 'A4 + adjective-noun pairs = 657 patterns (w2v)'),
         ('A total: final vs base Alg.2',     F_, 'A0 base Alg.2, abstract nouns kept (w2v)'),
         ('B1 drop pair patterns',            'B1 final, adjective patterns only', F_),
         ('B2 drop adjective patterns',       'B2 final, pair patterns only', F_),
         ('B3 log-lift weighting',            'B3 final, log-lift instead of confidence', F_),
         ('B4 frequentist EN selection',      'B4 final, frequentist EN (C=0.5)', F_),
         ('B5 Bayesian EN selection',         'B5 final, Bayesian EN', F_),
         ('B6 drop the table',                'B6 final, nouns only (no table)', F_),
         ('B7 naive adjectives, no table',    'B7 final, adjectives appended instead of table', F_),
         ('D small parser',                   'D final, small parser (en_core_web_sm)', F_)]
PAIRS += [(f'C {n[2:]} vs 0.0005', n, 'C support 0.0005') for n in SUPPORT if n != 'C support 0.0005']
CMP = []
for label, a, b in PAIRS:
    ha, hb = RANKS[a]['pk_ext'] <= 3, RANKS[b]['pk_ext'] <= 3
    d, lo, hi = boot(ha, hb); n01, n10 = int((~ha & hb).sum()), int((ha & ~hb).sum())
    CMP.append({'comparison': label, 'set': 'pk_ext', 'a': a, 'b': b, 'delta_hits3': d, 'ci95': [lo, hi], 'a_only': n10, 'b_only': n01,
                'p': float(binomtest(min(n01, n10), n01 + n10, 0.5).pvalue) if n01 + n10 else 1.0})
ps = np.array([c['p'] for c in CMP]); o = np.argsort(ps); m_ = len(ps); adj = np.empty(m_); run = 0.0
for rank_, i in enumerate(o): run = max(run, min(1.0, (m_ - rank_) * ps[i])); adj[i] = run
for c, pa in zip(CMP, adj): c['p_holm'] = float(pa)
for c in CMP: print(f"{c['comparison']:34s} dH@3 {c['delta_hits3']:+.3f} [{c['ci95'][0]:+.3f},{c['ci95'][1]:+.3f}] p_holm {c['p_holm']:.4f}")
json.dump({'status': 'post-hoc ablation on the extended Pakistan set (528 rows, extractor nouns); every arm reproduces ablation.json on OSM dev and the 106 set',
           'lambda_rule': 'per arm, chosen on OSM dev (as in ablation.json), applied unchanged to pk_ext',
           'sets': {s: len(v) for s, v in SETS.items()}, 'arms': {n: {'lambda': a['lambda'], 'pk_ext': a['pk_ext']} for n, a in ARMS.items()},
           'support_sensitivity': SUPPORT, 'comparisons': CMP, 'holm_family_size': m_, 'extra_query_vectors': {'w2v': len(NEW_W), 'numberbatch': len(NEW_N)},
           'not_scored': 'F arms (Wikivoyage tables): the extended rows\' source listings are in that training text'},
          open(f'{OUT}/ablation_pk_extended.json', 'w'), indent=1, default=str)
print('wrote', f'{OUT}/ablation_pk_extended.json')
